# Decisive test of the drift-budget law — `p* ≈ (η·‖ḡ‖) / R`

This notebook runs **one decisive experiment** from a cold Colab start and produces an
honest yes/no verdict on whether the closed-form prediction holds:

> Minimum tether strength to avoid collapse scales linearly with the adaptation-signal
> gradient norm: **p\* ≈ (η · ‖ḡ‖) / R**, with **R** a per-architecture constant.

Hold architecture fixed, vary CIFAR-10-C **severity** (the shift-magnitude axis). If the
law holds, **p\* vs (η·‖ḡ‖)** is a straight line through the origin (slope = 1/R) — one
line per architecture (`resnet18`, `wrn28_10`).

**All real logic lives in version-controlled scripts** (`scripts/run_pstar_sweep.py`,
`scripts/analyze_pstar_law.py`, `scripts/pstar_common.py`). This notebook only: sets up
→ fetches data/checkpoints → validates the pipeline → calls the scripts → shows results.

By default `wrn28_10` (the decisive, sharp-boundary line) gets full severity density
`[1,2,3,4,5]` and `resnet18` gets the cheaper `[1,3,5]` — 8 cells total. Both are
configurable.

Run top-to-bottom. The sweep is **resumable**: if Colab disconnects, just re-run the
sweep cell — completed runs are skipped. See `notebooks/README_pstar.md` first.

## 1. GPU check

In [ ]:
# Print the GPU. A GPU is strongly recommended (the sweep is ~hours on CPU).
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode == 0:
    print(out.stdout)
else:
    print("⚠️  No GPU detected. Set Runtime → Change runtime type → GPU (L4/A100).")
    print("    The experiment will run on CPU but will be far slower than the quoted times.")

## 2. Config — `# === EDIT ME ===`

Everything you might change is here. Defaults run `resnet18` at severities `[1,3,5]` and
`wrn28_10` at `[1,2,3,4,5]` (8 cells; WRN is the decisive line and gets full density).
That is roughly **~7–9 h on an L4**, faster on A100. To run the cheaper reduced pass for
both archs, set `SEVERITIES = [1, 3, 5]` (a plain list applies to every arch).

In [ ]:
# === EDIT ME ===========================================================
# --- Repo ---
REPO_URL  = "https://github.com/octadion/heat.git"   # or upload the repo manually
REPO_DIR  = "heat"                                    # folder name after clone
GIT_BRANCH = ""                                       # "" = default branch

# --- Where results live (Drive-backed so they survive a disconnect) ---
USE_DRIVE         = True
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/pstar_results"
LOCAL_RESULTS_DIR = "/content/pstar_results"          # used only if USE_DRIVE = False

# --- Source checkpoints ---
# Each may be: a direct http(s) URL (wget), a Google-Drive file id or share URL
# (gdown), OR a local path already on disk. Leave "" to TRAIN from scratch
# (requires TRAIN_IF_MISSING = True). Filenames are normalized to
# experiments/checkpoints/<arch>_final.pt.
CKPT_RESNET18    = ""
CKPT_WRN         = ""
TRAIN_IF_MISSING = False           # WRN trains ~4x slower than ResNet
TRAIN_EPOCHS     = 30

# --- Experiment grid ---
ARCHS       = ["resnet18", "wrn28_10"]   # resnet18 is processed first (cheaper)
# Per-architecture severity grids (the shift-magnitude axis). WRN is the decisive
# line and gets full density; ResNet uses the cheaper reduced set. Set SEVERITIES
# to a plain list (e.g. [1, 3, 5]) to apply the SAME severities to every arch.
SEVERITIES  = {"resnet18": [1, 3, 5], "wrn28_10": [1, 2, 3, 4, 5]}
P_GRID      = [0.0, 0.005, 0.010, 0.020]  # coarse tether grid
BISECT_STEPS = 2                          # bisections of the stable/collapse bracket
SEED        = 42

# --- Data / loader ---
C10C_ROOT   = "data/cifar10c"   # relative to the repo dir (we cd into it)
BATCH_SIZE  = 64
NUM_WORKERS = 2
# =======================================================================

RESULTS_DIR = DRIVE_RESULTS_DIR if USE_DRIVE else LOCAL_RESULTS_DIR
# The WRN checkpoint is always needed for the mandatory pipeline-validation gate.
CKPT_ARCHS = sorted(set(ARCHS) | {"wrn28_10"})

def sev_list(arch):
    """Severities for one arch (handles both the per-arch dict and a plain list)."""
    return SEVERITIES[arch] if isinstance(SEVERITIES, dict) else list(SEVERITIES)

# Union across archs — what the analysis step scans (it recovers only cells that
# actually have runs, so the union is safe to pass).
ALL_SEVERITIES = sorted({s for a in ARCHS for s in sev_list(a)})

print("RESULTS_DIR =", RESULTS_DIR)
print("Archs needing a checkpoint (incl. WRN for the validation gate):", CKPT_ARCHS)
print("Severities per arch:", {a: sev_list(a) for a in ARCHS})

## 3. Mount Google Drive (if `USE_DRIVE`)

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results will be written to:", RESULTS_DIR)
print("(Re-running the sweep cell after a disconnect resumes from here.)")

## 4. Clone the repo and install dependencies

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    cmd = ["git", "clone"]
    if GIT_BRANCH:
        cmd += ["--branch", GIT_BRANCH]
    cmd += [REPO_URL, REPO_DIR]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print(f"[skip clone] {REPO_DIR}/ already exists.")

os.chdir("/content/" + REPO_DIR if not os.path.isabs(REPO_DIR) else REPO_DIR)
REPO_ROOT = os.getcwd()
print("cwd =", REPO_ROOT)

# Most deps are preinstalled in Colab; this is a no-op / fast top-up. gdown for
# Drive-hosted checkpoints.
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run(["pip", "install", "-q", "gdown"], check=False)

os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH =", os.environ["PYTHONPATH"])

## 5. Get CIFAR-10-C (all 5 severities)

Uses the repo's downloader (canonical Zenodo tar, ~2.5 GB). We then verify that
**all five severities** are actually present, not just severity 5.

In [ ]:
import subprocess, numpy as np, os
subprocess.run(["python", "scripts/download_cifar10c.py", "--root", C10C_ROOT], check=True)

# Verify: each corruption .npy must hold 50000 = 5 severities x 10000 images.
labels_path = os.path.join(C10C_ROOT, "labels.npy")
assert os.path.exists(labels_path), f"labels.npy missing in {C10C_ROOT}"
n_labels = len(np.load(labels_path))
assert n_labels == 50000, f"expected 50000 labels (5 severities), got {n_labels}"

probe = os.path.join(C10C_ROOT, "gaussian_noise.npy")
n_imgs = np.load(probe, mmap_mode="r").shape[0]
assert n_imgs == 50000, f"gaussian_noise.npy has {n_imgs} rows, expected 50000 (5 severities)"
print(f"[ok] CIFAR-10-C present with all 5 severities ({n_imgs} imgs/corruption, "
      f"{n_labels} labels).")

## 6. Get the source checkpoints

For each architecture: download (wget / gdown) or train. Resolved paths are echoed.

In [ ]:
import os, subprocess, shutil

def _looks_like_url(s):
    return s.startswith("http://") or s.startswith("https://")

def resolve_ckpt(arch, spec):
    dest = f"experiments/checkpoints/{arch}_final.pt"
    os.makedirs("experiments/checkpoints", exist_ok=True)
    if os.path.exists(dest):
        print(f"[skip] {arch}: {dest} already present.")
        return dest
    if spec:
        if os.path.exists(spec):
            print(f"[local] {arch}: using {spec}")
            return spec
        if _looks_like_url(spec) and "drive.google.com" not in spec:
            print(f"[wget] {arch}: {spec}")
            subprocess.run(["wget", "-q", "-O", dest, spec], check=True)
        else:
            # Treat as a gdown id or Drive share URL.
            print(f"[gdown] {arch}: {spec}")
            if _looks_like_url(spec):
                subprocess.run(["gdown", "--fuzzy", "-O", dest, spec], check=True)
            else:
                subprocess.run(["gdown", "--id", spec, "-O", dest], check=True)
        return dest
    if TRAIN_IF_MISSING:
        print(f"[train] {arch}: training source model ({TRAIN_EPOCHS} epochs)"
              + (" — WRN is ~4x slower" if arch == "wrn28_10" else ""))
        subprocess.run(["python", "scripts/train_source.py", "--arch", arch,
                        "--epochs", str(TRAIN_EPOCHS)], check=True)
        return dest  # train_source writes experiments/checkpoints/<arch>_final.pt
    raise SystemExit(
        f"[fatal] no checkpoint for {arch}. Set CKPT_{'RESNET18' if arch=='resnet18' else 'WRN'} "
        f"or TRAIN_IF_MISSING = True.")

_spec = {"resnet18": CKPT_RESNET18, "wrn28_10": CKPT_WRN}
CKPT_PATHS = {arch: resolve_ckpt(arch, _spec.get(arch, "")) for arch in CKPT_ARCHS}
print("\nResolved checkpoints:")
for a, p in CKPT_PATHS.items():
    print(f"  {a:10s} -> {p}  (exists={os.path.exists(p)})")

## 7. Pipeline validation (mandatory gate)

Reproduce the known **WRN-28-10 severity-5 anchor** before trusting anything:
`p=0 → NaN ~step 603`, `p=0.005 → NaN ~step 920`, `p≥0.010 → stable` with stationary
‖ḡ‖ ≈ 13. **If this does not reproduce, the cell stops and you must not proceed.**
(These runs are part of the real sweep, so nothing is wasted — they are resumed below.)

In [ ]:
import subprocess
rc = subprocess.run([
    "python", "scripts/run_pstar_sweep.py", "--sanity-check-only",
    "--results-dir", RESULTS_DIR,
    "--ckpt-wrn", CKPT_PATHS["wrn28_10"],
    "--c10c-root", C10C_ROOT,
    "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]).returncode

assert rc == 0, (
    "PIPELINE VALIDATION FAILED — the known WRN sev-5 collapse anchor did NOT "
    "reproduce (see output above). Do NOT run the sweep. Check: correct WRN "
    "checkpoint, all 5 severities downloaded, GPU active.")
print("\n✅ Pipeline validated. Proceeding to the sweep is safe.")

## 8. Run the sweep (resumable)

Loops (arch, severity, p), calls `run_tier2.py --protocol p9` for each, applies the
collapse criterion, and bisects the stable/collapse bracket to localize p*. Each run's
JSON lands in `RESULTS_DIR` immediately. **Re-run this cell after any disconnect** — it
skips completed runs and continues. `resnet18` is processed first.

In [ ]:
import subprocess
cmd = [
    "python", "scripts/run_pstar_sweep.py",
    "--results-dir", RESULTS_DIR,
    "--c10c-root", C10C_ROOT,
    "--archs", *ARCHS,
    "--p-grid", *[repr(p) for p in P_GRID],
    "--bisect-steps", str(BISECT_STEPS),
    "--seed", str(SEED),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]
# Severities: a per-arch dict -> per-arch flags; a plain list -> one global flag.
if isinstance(SEVERITIES, dict):
    for arch in ARCHS:
        cmd += [f"--severities-{arch}", *[str(s) for s in sev_list(arch)]]
else:
    cmd += ["--severities", *[str(s) for s in SEVERITIES]]
if "resnet18" in CKPT_PATHS:
    cmd += ["--ckpt-resnet18", CKPT_PATHS["resnet18"]]
if "wrn28_10" in CKPT_PATHS:
    cmd += ["--ckpt-wrn", CKPT_PATHS["wrn28_10"]]
print("$", " ".join(cmd), "\n")
subprocess.run(cmd, check=True)

## 9. Analyze — fit the line, plot, verdict

Reads every run JSON, measures ‖ḡ‖ and p* per (arch, severity), fits the per-architecture
line, and writes `analysis/pstar_law.{json,png}` + `pstar_verdict.md`. The plot and the
verdict are displayed **inline** below.

In [ ]:
import subprocess, os
from IPython.display import Image, Markdown, display

subprocess.run([
    "python", "scripts/analyze_pstar_law.py",
    "--results-dir", RESULTS_DIR,
    "--archs", *ARCHS,
    "--severities", *[str(s) for s in ALL_SEVERITIES],   # union across archs
    "--seed", str(SEED),
], check=True)

png = os.path.join(RESULTS_DIR, "analysis", "pstar_law.png")
verdict = os.path.join(RESULTS_DIR, "analysis", "pstar_verdict.md")
if os.path.exists(png):
    display(Image(filename=png))
if os.path.exists(verdict):
    display(Markdown(open(verdict, encoding="utf-8").read()))